In [1]:
pip install requests transformers datasets einops peft scikit-learn xgboost

Defaulting to user installation because normal site-packages is not writeable


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import sklearn
import requests

In [1]:
import torch
import json, requests

from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, TrainingArguments, Trainer

from datasets import Dataset
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

hf_cache = "/scratch/sguirale/tmp"

/users/sguirale/miniconda3/envs/savant2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Get Training Data

In [4]:
raw_df = pd.read_csv("../ML_model/combined_50k_filtered.csv")

raw_df.drop(columns=["Job_ID", "molecular_weight", "logP", "num_h_donors", "num_h_acceptors", "num_rotatable_bonds", "tpsa", "num_rings", "num_aromatic_rings", "num_atoms", "num_heavy_atoms", "formal_charge", "score", "total", "vdw", "elec", "desolv", "bsa", "num_interacting_residues", "iptm" ], inplace=True)

raw_df.head()

,seq,smiles,neg_log10_affinity_M
0,MTVPDRSEIAGKWYVVALASNTEFFLREKDKMKMAMARISFLGEDE...,CCCCCCCCCCCCCCCCCCCC(=O)O,7.585027
1,APQTITELCSEYRNTQIYTINDKILSYTESMAGKREMVIITFKSGE...,OC[C@H]1O[C@H](Oc2cccc(c2)N(=O)=O)[C@@H]([C@H]...,3.301030
2,VETFAFQAEIAQLMSLIINTFYSNKEIFLRELISNSSDALDKIRYE...,COc1ccc(cc1)c1c(onc1c1cc(C(C)C)c(cc1O)O)NC(=O)...,7.638272
3,AAPFDKSKNVAQSIDQLIGQTPALYLNKLNNTKAKVVLKMECENPM...,OC[C@@H](C(=O)N[C@@H]([C@H](CC)C)C(=O)O)NC(=O)...,5.191789
4,YITFRSFTAVLIAFFLTLVLSPSFINRLRKIQRKKYTPTMGGIVIL...,CO[C@@H]1[C@H](O[C@H]([C@@H]1O)n1ccc(=O)[nH]c1...,6.732828


In [5]:
## Other Filters

raw_df = raw_df[raw_df['neg_log10_affinity_M'] >= 4]


print(f"Filtered dataset size: {len(raw_df)}")

Filtered dataset size: 44740


## Load Protein Tokenizer


In [6]:
## Load Protein Models
base_model_path = "Synthyra/ESM2-8M"

esm2_model = AutoModel.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    attn_implementation="sdpa",
    cache_dir=hf_cache
).to(device).eval()


esm2_tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    cache_dir=hf_cache
)

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

[transformers] FastEsmModel LOAD REPORT from: Synthyra/ESM2-8M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
## Function to get sequence embeddings
@torch.inference_mode()
def get_sequence_embeddings(model, sequences, batch_size=4):
    embedding_batches = []

    for start in range(0, len(sequences), batch_size):
        sequence_batch = sequences[start:start + batch_size]
        batch = esm2_tokenizer(
            sequence_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [8]:
aa_sequences = raw_df['seq'].tolist()

In [9]:
aa_emb = get_sequence_embeddings(esm2_model, aa_sequences)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


## Get Ligand Tokenizer

In [10]:
ligand_model_name = "DeepChem/ChemBERTa-77M-MLM"

ligand_tokenizer = AutoTokenizer.from_pretrained(ligand_model_name, cache_dir=hf_cache)
ligand_model = AutoModel.from_pretrained(ligand_model_name, cache_dir=hf_cache).to(device).eval()

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
## Function to get SMILES embeddings
@torch.inference_mode()
def get_smiles_embeddings(model, smiles, batch_size=32):
    embedding_batches = []

    for start in range(0, len(smiles), batch_size):
        smiles_batch = smiles[start:start + batch_size]
        batch = ligand_tokenizer(
            smiles_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [12]:
smiles_sequences = raw_df['smiles'].tolist()

In [13]:
lig_emb = get_smiles_embeddings(ligand_model, smiles_sequences)

## Train Model

In [14]:
## Make dataset of embeddings and neg_log10_affinity_M values
ligand_features = lig_emb.numpy()
aa_features = aa_emb.numpy()

if len(ligand_features) != len(aa_features) or len(ligand_features) != len(raw_df):
    raise ValueError("Ligand embeddings, protein embeddings, and targets must have the same number of rows")

## One row per protein-ligand pair, with both embeddings as numeric features.
X = np.concatenate([ligand_features, aa_features], axis=1)
y = pd.to_numeric(raw_df['neg_log10_affinity_M'], errors='coerce').to_numpy(dtype=np.float32)

valid_rows = np.isfinite(y)
X = X[valid_rows]
y = y[valid_rows]

print(f"Features: {X.shape}; targets: {y.shape}")

Features: (44740, 704); targets: (44740,)


In [15]:
## Inspect target and embedding coverage
print(raw_df['neg_log10_affinity_M'].describe())
print(f"Unique protein sequences: {raw_df['seq'].nunique()}")
print(f"Unique ligands: {raw_df['smiles'].nunique()}")
print(f"Duplicate protein-ligand pairs: {raw_df.duplicated(['seq', 'smiles']).sum()}")
print(f"NaN or infinite features: {np.isnan(X).sum() + np.isinf(X).sum()}")

count    44740.000000
mean         6.665419
std          1.506035
min          4.000000
25%          5.468521
50%          6.621420
75%          7.732828
max         15.000000
Name: neg_log10_affinity_M, dtype: float64
Unique protein sequences: 11222
Unique ligands: 30309
Duplicate protein-ligand pairs: 1223


NaN or infinite features: 0


In [16]:
## Turnover is usually strongly right-skewed; fit the model in log space.
y_log = np.log1p(y)
# y_log = np.log10(y+1e-8)  # Add a small constant to avoid log(0)

In [17]:
## Make train/validation/test splits
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    # y,
    y_log,
    test_size=0.2,
    random_state=1337
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=1337
)


print(f"Train: {len(y_train)}, validation: {len(y_valid)}, test: {len(y_test)}")

Train: 26844, validation: 8948, test: 8948


In [18]:
## Train Turnover Model
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=3000,
    max_depth=25,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=50,
    random_state=1337
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [19]:
## Evaluate Turnover Model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

predictions_log = model.predict(X_test)

## For log1p
predictions = np.maximum(np.expm1(predictions_log), 0)
y_test_original = np.expm1(y_test)
y_train_original = np.expm1(y_train)

## For log10
# predictions = np.maximum(np.power(10, predictions_log) - 1e-8, 0)
# y_test_original = np.power(10, y_test) - 1e-8
# y_train_original = np.power(10, y_train) - 1e-8

## Evaluate in log space, the scale used for training.
log_rmse = np.sqrt(mean_squared_error(y_test, predictions_log))
log_mae = mean_absolute_error(y_test, predictions_log)
log_r2 = r2_score(y_test, predictions_log)

## Evaluate in original turnover units for interpretability.
rmse = np.sqrt(mean_squared_error(y_test_original, predictions))
mae = mean_absolute_error(y_test_original, predictions)
r2 = r2_score(y_test_original, predictions)

## Spearman correlation
spearman_corr, spearman_p = spearmanr(y_test_original, predictions)

## Median baselines on the matching target scales.
log_baseline = np.full_like(y_test, np.median(y_train))
original_baseline = np.full_like(y_test_original, np.median(y_train_original))

log_baseline_rmse = np.sqrt(mean_squared_error(y_test, log_baseline))
baseline_rmse = np.sqrt(mean_squared_error(y_test_original, original_baseline))
baseline_mae = mean_absolute_error(y_test_original, original_baseline)

print(f"Log RMSE: {log_rmse:.4f} (median baseline: {log_baseline_rmse:.4f})")
print(f"Log MAE:  {log_mae:.4f}")
print(f"Log R^2:  {log_r2:.4f}")
print(f"Original RMSE: {rmse:.4f} (median baseline: {baseline_rmse:.4f})")
print(f"Original MAE:  {mae:.4f} (median baseline: {baseline_mae:.4f})")
print(f"Spearman correlation: {spearman_corr:.4f} (p-value: {spearman_p:.4f})")
print(f"Original R^2:  {r2:.4f}")
print(f"Best iteration: {model.best_iteration}")

Log RMSE: 0.1607 (median baseline: 0.1985)
Log MAE:  0.1271
Log R^2:  0.3393
Original RMSE: 1.2083 (median baseline: 1.4989)
Original MAE:  0.9531 (median baseline: 1.2326)
Spearman correlation: 0.5854 (p-value: 0.0000)
Original R^2:  0.3502
Best iteration: 178


In [20]:
## Compare with a regularized linear model
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10.0)
)
ridge_model.fit(X_train, y_train)
ridge_predictions_log = ridge_model.predict(X_test)
ridge_predictions = np.maximum(np.expm1(ridge_predictions_log), 0)

ridge_log_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions_log))
ridge_log_mae = mean_absolute_error(y_test, ridge_predictions_log)
ridge_log_r2 = r2_score(y_test, ridge_predictions_log)

ridge_rmse = np.sqrt(mean_squared_error(y_test_original, ridge_predictions))
ridge_mae = mean_absolute_error(y_test_original, ridge_predictions)
ridge_r2 = r2_score(y_test_original, ridge_predictions)

print(f"Ridge log RMSE: {ridge_log_rmse:.4f}")
print(f"Ridge log MAE:  {ridge_log_mae:.4f}")
print(f"Ridge log R^2:  {ridge_log_r2:.4f}")
print(f"Ridge original RMSE: {ridge_rmse:.4f}")
print(f"Ridge original MAE:  {ridge_mae:.4f}")
print(f"Ridge original R^2:  {ridge_r2:.4f}")

Ridge log RMSE: 0.1774
Ridge log MAE:  0.1438
Ridge log R^2:  0.1942
Ridge original RMSE: 1.3428
Ridge original MAE:  1.0822
Ridge original R^2:  0.1974
